# Diffusion Models na prática: DDPM do zero + Stable Diffusion
**SENAI CIMATEC · IA Generativa** · Bianca Andrade, Rafael Athaliba, Pedro Lima

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/biaag10/machine_learning_projects/blob/develop/diffusion_models/notebooks/Diffusion_Models_MNIST_SD.ipynb)

Este notebook cobre os 5 itens do trabalho:

| Item | Onde está |
|---|---|
| 1. Fundamentação | Seção 3 (processo direto) e comentários das funções |
| 2. Arquitetura | Seção 4 (U-Net) + diagramas em `reports/figures/` |
| 3. Implementação funcional | Seções 1–7 (DDPM treinado do zero no MNIST) |
| 4. Avaliação experimental | Seções 6, 8, 9 e 10 (curva de loss, acurácia condicional, FID, tempo, erros, diversidade) |
| 5. Demonstração | Seção 11 (app Gradio do nosso modelo) e Seção 12 (Stable Diffusion, texto → imagem) |

### Como rodar
1. Clique no botão **Open in Colab** acima.
2. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.
3. `Ambiente de execução → Executar tudo`.

Por padrão (`TREINAR_DO_ZERO = False`, Seção 1), o notebook **carrega o modelo já treinado** do repositório
(`models/ddpm_mnist_ema.pt`, o mesmo dos slides) e não precisa treinar de novo.
Marque `TREINAR_DO_ZERO = True` para refazer o treino (cerca de 10 min na T4 ou 95 min em CPU).

As figuras geradas são salvas na pasta `figuras/` (no Colab, no painel de arquivos à esquerda).

In [ ]:
#@title 0. Instalação das dependências (≈1 min no Colab)
# No Colab instalamos o que falta; fora dele, use: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip -q install "torchmetrics[image]" gradio diffusers transformers accelerate

In [ ]:
#@title 1. Imports, GPU, reprodutibilidade e configurações
import math, time, os, copy, urllib.request
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "SEM GPU — ative a GPU no Colab!")
torch.manual_seed(42); np.random.seed(42)
FIG_DIR = "figuras"
os.makedirs(FIG_DIR, exist_ok=True)

def salvar(nome):
    """Salva a figura atual em FIG_DIR/<nome>.png (para usar nos slides)."""
    plt.savefig(f"{FIG_DIR}/{nome}.png", dpi=200, bbox_inches="tight", facecolor="white")

def show(imgs, nrow=10, title=None, size=10, nome=None):
    """Mostra um lote de imagens no intervalo [-1, 1] (e salva se receber um nome)."""
    grid = torchvision.utils.make_grid((imgs.cpu() + 1) / 2, nrow=nrow, padding=2, pad_value=1)
    plt.figure(figsize=(size, size * grid.shape[1] / grid.shape[2]))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1), interpolation="nearest"); plt.axis("off")
    if title: plt.title(title)
    if nome: salvar(nome)
    plt.show()

# =============================================
# CONFIGURAÇÕES DO EXPERIMENTO
# =============================================
TREINAR_DO_ZERO = False  #@param {type:"boolean"}
# False: carrega o modelo já treinado (o mesmo dos slides). True: treina de novo (Seção 5).

CH = 32  #@param [32, 64] {type:"raw"}
# Largura da U-Net: 32 -> ~0,75 M parâmetros (nosso experimento); 64 -> ~3 M (melhor, use com GPU).

EPOCHS = 10  #@param {type:"integer"}

# Avaliação (Seção 8): nos slides usamos 500 imagens por configuração, em CPU (~2 h no total).
# Em CPU usamos 100 por padrão para a avaliação caber em poucos minutos.
N_EVAL = 500 if device == "cuda" else 100
AVALIAR_DDPM_1000 = device == "cuda"   # o DDPM-1000 é ~20x mais lento que o DDIM-50

MODEL_PATH = Path("ddpm_mnist_ema.pt")
MODEL_URL = "https://raw.githubusercontent.com/biaag10/machine_learning_projects/develop/diffusion_models/models/ddpm_mnist_ema.pt"

if not TREINAR_DO_ZERO and CH != 32:
    print("O modelo pronto foi treinado com CH = 32; usando CH = 32.")
    CH = 32

## 2. Dados — MNIST
Usamos o MNIST (70 mil dígitos manuscritos 28×28). Redimensionamos para **32×32** (fica divisível por 2 nas descidas da U-Net)
e normalizamos para **[-1, 1]**, a mesma escala do ruído gaussiano.

In [ ]:
#@title 2. Carregando o MNIST
tf = transforms.Compose([transforms.Resize(32), transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
train_ds = torchvision.datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds  = torchvision.datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
print(len(train_ds), "imagens de treino |", len(test_ds), "de teste")
xb, yb = next(iter(train_dl))
show(xb[:20], title="Amostras reais do MNIST", nome="01_mnist_real")

## 3. Fundamentação — o processo direto (forward)
Adicionamos ruído gaussiano aos poucos, ao longo de **T = 1000** passos, com variâncias $\beta_t$ crescentes (1e-4 → 0,02).
Graças a uma propriedade da gaussiana, dá para pular direto de $x_0$ para qualquer $x_t$:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I),\quad \bar\alpha_t=\prod_{s\le t}(1-\beta_s)$$

Esse processo **não tem nada a aprender** — é só matemática. Quem aprende é a rede que faz o caminho inverso.

In [ ]:
#@title 3. Agendamento de ruído (noise schedule) e processo direto q(x_t | x_0)
T = 1000                                            # nº de passos de difusão
betas = torch.linspace(1e-4, 0.02, T, device=device)  # variância do ruído em cada passo (linear, Ho et al. 2020)
alphas = 1.0 - betas
alphas_bar = torch.cumprod(alphas, dim=0)            # ᾱ_t = produto de α_s até t
alphas_bar_prev = torch.cat([torch.ones(1, device=device), alphas_bar[:-1]])
# variância da posterior q(x_{t-1} | x_t, x_0)
posterior_var = betas * (1 - alphas_bar_prev) / (1 - alphas_bar)

def q_sample(x0, t, noise=None):
    """Processo DIRETO: pula direto de x_0 para x_t em um único passo.
    x_t = sqrt(ᾱ_t)·x_0 + sqrt(1-ᾱ_t)·ε ,  ε ~ N(0, I)"""
    if noise is None:
        noise = torch.randn_like(x0)
    ab = alphas_bar[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

# Visualizando o processo direto em um dígito
x0 = xb[:1].to(device)
steps_show = [0, 50, 100, 200, 300, 500, 700, 999]
imgs = torch.cat([q_sample(x0, torch.tensor([t], device=device)) for t in steps_show])
show(imgs, nrow=len(steps_show), title="Processo direto: t = " + ", ".join(map(str, steps_show)), nome="02_processo_direto")

plt.figure(figsize=(6, 3))
plt.plot(alphas_bar.cpu(), label="ᾱ_t (fração de sinal)")
plt.plot((1 - alphas_bar).cpu(), label="1-ᾱ_t (fração de ruído)")
plt.xlabel("passo t"); plt.legend(); plt.title("Quanto sobra de sinal x ruído"); salvar("03_sinal_x_ruido"); plt.show()

## 4. Arquitetura — U-Net condicionada
A rede recebe **(imagem ruidosa $x_t$, passo $t$, classe $y$)** e devolve **o ruído previsto $\hat\varepsilon$**.

- **Encoder** (32→16→8) extrai contexto; **decoder** (8→16→32) reconstrói o detalhe; **skip connections** ligam os dois lados.
- **Embedding senoidal de tempo**: diz à rede “quanto ruído existe agora”.
- **Embedding de classe**: diz “qual dígito eu quero”. A classe extra `10` significa “sem condição” — usada no *Classifier-Free Guidance*.
- **Self-attention** na resolução 8×8 para relações globais.

**Tamanho do modelo (`CH`)**: `32` → ~0,75 M parâmetros (configuração do nosso experimento, roda até sem GPU); `64` → ~3 M parâmetros (melhor qualidade, use com GPU).

In [ ]:
#@title 4. Arquitetura: U-Net com embedding de tempo e de classe
class SinusoidalEmb(nn.Module):
    """Transforma o passo t (inteiro) em um vetor, igual ao positional encoding do Transformer."""
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.float()[:, None] * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=1)

class ResBlock(nn.Module):
    """Bloco residual que recebe também o embedding (tempo + classe)."""
    def __init__(self, cin, cout, emb_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, cin);  self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.emb = nn.Linear(emb_dim, cout)
        self.norm2 = nn.GroupNorm(8, cout); self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb(F.silu(emb))[:, :, None, None]   # injeta "em que passo estou" e "qual dígito quero"
        h = self.conv2(self.drop(F.silu(self.norm2(h))))
        return h + self.skip(x)

class Attention(nn.Module):
    """Self-attention espacial (usado na resolução 8x8)."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.attn = nn.MultiheadAttention(ch, num_heads=4, batch_first=True)
    def forward(self, x):
        b, c, h, w = x.shape
        y = self.norm(x).flatten(2).transpose(1, 2)        # (B, H*W, C)
        y, _ = self.attn(y, y, y, need_weights=False)
        return x + y.transpose(1, 2).reshape(b, c, h, w)

class UNet(nn.Module):
    """Entrada: imagem ruidosa x_t (1x32x32), passo t e rótulo y.  Saída: ruído previsto ε̂ (1x32x32)."""
    def __init__(self, ch=64, num_classes=10):
        super().__init__()
        emb_dim = ch * 4
        self.null_class = num_classes                       # índice extra = "sem condição" (para CFG)
        self.time_emb = nn.Sequential(SinusoidalEmb(ch), nn.Linear(ch, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(num_classes + 1, emb_dim)
        self.inc = nn.Conv2d(1, ch, 3, padding=1)
        # ---- Encoder (descida) ----
        self.d1 = ResBlock(ch, ch, emb_dim);          self.down1 = nn.Conv2d(ch, ch, 3, stride=2, padding=1)        # 32 -> 16
        self.d2 = ResBlock(ch, ch * 2, emb_dim);      self.down2 = nn.Conv2d(ch * 2, ch * 2, 3, stride=2, padding=1) # 16 -> 8
        self.d3 = ResBlock(ch * 2, ch * 2, emb_dim);  self.a3 = Attention(ch * 2)
        # ---- Gargalo ----
        self.m1 = ResBlock(ch * 2, ch * 2, emb_dim);  self.ma = Attention(ch * 2)
        self.m2 = ResBlock(ch * 2, ch * 2, emb_dim)
        # ---- Decoder (subida) com skip connections ----
        self.u3 = ResBlock(ch * 4, ch * 2, emb_dim);  self.ua3 = Attention(ch * 2)
        self.up2 = nn.Upsample(scale_factor=2, mode="nearest")                                                      # 8 -> 16
        self.u2 = ResBlock(ch * 4, ch * 2, emb_dim)
        self.up1 = nn.Upsample(scale_factor=2, mode="nearest")                                                      # 16 -> 32
        self.u1 = ResBlock(ch * 3, ch, emb_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, 1, 3, padding=1))

    def forward(self, x, t, y):
        emb = self.time_emb(t) + self.class_emb(y)
        h0 = self.inc(x)
        s1 = self.d1(h0, emb)                         # 64  x 32x32
        s2 = self.d2(self.down1(s1), emb)             # 128 x 16x16
        s3 = self.a3(self.d3(self.down2(s2), emb))    # 128 x 8x8
        h = self.m2(self.ma(self.m1(s3, emb)), emb)
        h = self.ua3(self.u3(torch.cat([h, s3], 1), emb))
        h = self.u2(torch.cat([self.up2(h), s2], 1), emb)
        h = self.u1(torch.cat([self.up1(h), s1], 1), emb)
        return self.out(h)

model = UNet(ch=CH).to(device)
print(f"Parâmetros: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

## 5. Treinamento
A cada iteração: sorteia $t$, sorteia o ruído $\varepsilon$, cria $x_t$ e treina a rede para acertar $\varepsilon$:

$$\mathcal L = \lVert \varepsilon - \varepsilon_\theta(x_t, t, y) \rVert^2$$

Em 10% das vezes trocamos a classe por “sem condição”, para que o mesmo modelo aprenda a gerar **com e sem** rótulo (necessário para o CFG).
Também mantemos uma **EMA** (média móvel dos pesos), que gera imagens mais limpas.

> **Modelo pronto:** com `TREINAR_DO_ZERO = False`, esta seção apenas carrega `ddpm_mnist_ema.pt`
> (da pasta `models/` do repositório, se o notebook estiver rodando dentro dele, ou baixando do GitHub).

In [ ]:
#@title 5. Treino do zero (≈1 min por época na T4) ou carregamento do modelo pronto
LR = 2e-4
P_UNCOND = 0.1       # probabilidade de "esconder" a classe (CFG)


def carregar_modelo_pronto():
    """Carrega os pesos (EMA) do modelo treinado: da pasta models/ do repositório ou do GitHub."""
    candidatos = [Path("../models/ddpm_mnist_ema.pt"), Path("models/ddpm_mnist_ema.pt"), MODEL_PATH]
    caminho = next((p for p in candidatos if p.exists()), None)
    if caminho is None:
        print("Baixando o modelo treinado do GitHub...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        caminho = MODEL_PATH
    rede = UNet(ch=CH).to(device)
    rede.load_state_dict(torch.load(caminho, map_location=device))
    print("Modelo carregado de", caminho)
    return rede.eval().requires_grad_(False)

losses = []
if TREINAR_DO_ZERO:
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    sched_lr = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.05)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    losses = []

    t_start = time.time()
    for epoch in range(EPOCHS):
        model.train(); ep_loss = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            y = torch.where(torch.rand(len(y), device=device) < P_UNCOND, torch.full_like(y, model.null_class), y)
            t = torch.randint(0, T, (len(x),), device=device)
            noise = torch.randn_like(x)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                loss = F.mse_loss(model(q_sample(x, t, noise), t, y), noise)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched_lr.step()
            with torch.no_grad():                                   # EMA dos pesos
                for pe, pm in zip(ema.parameters(), model.parameters()):
                    pe.lerp_(pm, 1 - 0.999)
            losses.append(loss.item()); ep_loss += loss.item()
        print(f"época {epoch+1:2d}/{EPOCHS} | loss médio {ep_loss/len(train_dl):.4f} | {time.time()-t_start:.0f}s")

    torch.save(ema.state_dict(), MODEL_PATH)
    print("Modelo salvo em", MODEL_PATH)
else:
    ema = carregar_modelo_pronto()

In [ ]:
#@title 6. Curva de perda (loss)
if len(losses):
    k = max(1, min(100, len(losses) // 5))   # janela da média móvel
    smooth = np.convolve(losses, np.ones(k) / k, mode="valid")
    plt.figure(figsize=(7, 3))
    plt.plot(losses, alpha=0.25, label="por iteração"); plt.plot(range(k - 1, len(losses)), smooth, label=f"média móvel ({k})")
    plt.yscale("log"); plt.xlabel("iteração"); plt.ylabel("MSE do ruído"); plt.legend(); plt.title("Treinamento"); salvar("05_curva_loss"); plt.show()
else:
    print("Modelo carregado pronto: a curva de loss do nosso treino está em reports/figures/05_curva_loss.png")
    print("(loss médio por época: 0,2084 → 0,0376 → ... → 0,0222 na 10ª época).")

## 7. Processo reverso — gerando imagens
- **DDPM** (Ho et al., 2020): 1000 passos estocásticos. Qualidade de referência, porém lento.
- **DDIM** (Song et al., 2021): determinístico e pula passos (ex.: 50). Mesmo modelo, sem retreinar.
- **Classifier-Free Guidance**: $\hat\varepsilon = \varepsilon_{\text{incond}} + w\,(\varepsilon_{\text{cond}} - \varepsilon_{\text{incond}})$. Quanto maior $w$, mais “fiel à classe” e menos variado.

In [ ]:
#@title 7. Amostradores: DDPM (1000 passos) e DDIM (poucos passos) com Classifier-Free Guidance
@torch.no_grad()
def predict_eps(model, x, t_int, y, guidance):
    """CFG: ε = ε_incond + w·(ε_cond − ε_incond).  w=0 → incondicional, w=1 → só condicional, w>1 → reforça a classe."""
    b = x.shape[0]
    t = torch.full((b,), t_int, device=x.device, dtype=torch.long)
    x_in = torch.cat([x, x]); t_in = torch.cat([t, t])
    y_in = torch.cat([y, torch.full_like(y, model.null_class)])
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(x.device.type == "cuda")):
        out = model(x_in, t_in, y_in).float()
    eps_c, eps_u = out.chunk(2)
    return eps_u + guidance * (eps_c - eps_u)

@torch.no_grad()
def sample_ddpm(model, y, guidance=3.0, save_every=None, seed=None):
    """Processo REVERSO estocástico (Ho et al., 2020): 1000 passos, x_T → x_0."""
    g = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
    model.eval()
    x = torch.randn(len(y), 1, 32, 32, device=device, generator=g)
    frames = []
    for t in reversed(range(T)):
        eps = predict_eps(model, x, t, y, guidance)
        ab = alphas_bar[t]
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)        # estimativa da imagem limpa
        mean = (alphas_bar_prev[t].sqrt() * betas[t] / (1 - ab)) * x0 \
             + (alphas[t].sqrt() * (1 - alphas_bar_prev[t]) / (1 - ab)) * x  # média da posterior
        if t > 0:
            x = mean + posterior_var[t].sqrt() * torch.randn(x.shape, device=device, generator=g)
        else:
            x = mean
        if save_every and (t % save_every == 0 or t == T - 1):
            frames.append((t, x.detach().cpu()))
    return x.clamp(-1, 1), frames

@torch.no_grad()
def sample_ddim(model, y, steps=50, guidance=3.0, save_every=None, seed=None):
    """Processo REVERSO determinístico (Song et al., 2021): pula passos, ex.: 1000 → 50."""
    g = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
    model.eval()
    x = torch.randn(len(y), 1, 32, 32, device=device, generator=g)
    ts = torch.linspace(T - 1, 0, steps).long().tolist()
    frames = []
    for i, t in enumerate(ts):
        eps = predict_eps(model, x, t, y, guidance)
        ab = alphas_bar[t]
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()                     # recalcula ε coerente com x0 recortado
        ab_prev = alphas_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=device)
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
        if save_every and (i % save_every == 0):
            frames.append((t, x.detach().cpu()))
    frames.append((0, x.detach().cpu()))
    return x.clamp(-1, 1), frames

# Grade: 10 classes x 8 amostras, DDIM 50 passos
y = torch.arange(10, device=device).repeat_interleave(8)
t0 = time.time(); imgs, _ = sample_ddim(ema, y, steps=50, guidance=3.0, seed=0)
print(f"DDIM-50: {time.time()-t0:.1f}s para {len(y)} imagens")
show(imgs.view(10, 8, 1, 32, 32).transpose(0, 1).reshape(-1, 1, 32, 32), nrow=10, title="Geradas (DDIM 50 passos, w=3) — cada coluna é um dígito", nome="06_grade_ddim50")

t0 = time.time(); imgs_ddpm, _ = sample_ddpm(ema, torch.arange(10, device=device), guidance=3.0, seed=0)
print(f"DDPM-1000: {time.time()-t0:.1f}s para 10 imagens")
show(imgs_ddpm, nrow=10, title="Geradas (DDPM 1000 passos, w=3)", nome="07_grade_ddpm1000")

In [ ]:
#@title 7b. Visualizando o processo reverso (ruído → dígito) + GIF
from PIL import Image
y = torch.arange(10, device=device)
_, frames = sample_ddpm(ema, y, guidance=3.0, save_every=100, seed=1)
chain = torch.cat([f[1] for f in frames])                     # (n_frames*10, 1, 32, 32)
show(chain, nrow=10, title="Processo reverso: cada linha é um instante t (999 → 0)", size=8, nome="08_processo_reverso")

# GIF para colocar no slide
_, frames = sample_ddpm(ema, y, guidance=3.0, save_every=20, seed=1)
gif = []
for t, x in frames:
    g = torchvision.utils.make_grid((x + 1) / 2, nrow=10, padding=2, pad_value=1)
    g = (g.permute(1, 2, 0).clamp(0, 1).numpy() * 255).astype(np.uint8)
    gif.append(Image.fromarray(g).resize((g.shape[1] * 3, g.shape[0] * 3), Image.NEAREST))
gif += [gif[-1]] * 15                                         # pausa no final
gif[0].save(f"{FIG_DIR}/09_processo_reverso.gif", save_all=True, append_images=gif[1:], duration=60, loop=0)
display(IPImage(f"{FIG_DIR}/09_processo_reverso.gif"))

## 8. Avaliação experimental — métricas
1. **Acurácia condicional**: treinamos um classificador CNN no MNIST real (~96% de acerto no nosso experimento) e perguntamos: *a imagem gerada para “7” é reconhecida como 7?*
2. **Confiança média** do classificador (nitidez/“legibilidade” do dígito).
3. **FID** (Fréchet Inception Distance): distância entre a distribuição de imagens reais e geradas. **Quanto menor, melhor.**
   Usamos `N_EVAL` imagens por configuração (500 nos slides) — o valor absoluto não é comparável com artigos (que usam 50 mil), mas serve para **comparar configurações entre si**.
4. **Tempo de geração**.

> A linha do DDPM-1000 é a mais demorada — é justamente o ponto que queremos mostrar. Todas as configurações partem do mesmo ruído inicial (mesma semente), então a comparação é justa.
>
> Em CPU, o DDPM-1000 fica fora da comparação por padrão (`AVALIAR_DDPM_1000`), porque levaria mais de 1 hora.

In [ ]:
#@title 8a. Classificador auxiliar (avaliador)
clf = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(128, 10)).to(device)
copt = torch.optim.Adam(clf.parameters(), 1e-3)
for ep in range(3):
    clf.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        l = F.cross_entropy(clf(x), y); copt.zero_grad(); l.backward(); copt.step()
clf.eval().requires_grad_(False)
test_dl = DataLoader(test_ds, batch_size=512)
correct = sum((clf(x.to(device)).argmax(1) == y.to(device)).sum().item() for x, y in test_dl)
print(f"Acurácia do classificador no teste real: {correct/len(test_ds):.2%}")

In [ ]:
#@title 8b. Comparação de amostradores, nº de passos e guidance
from torchmetrics.image.fid import FrechetInceptionDistance

def to_fid(x):          # [-1,1] 1 canal -> [0,1] 3 canais
    return ((x + 1) / 2).clamp(0, 1).repeat(1, 3, 1, 1)

# features reais calculadas uma única vez
fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)
real = torch.stack([test_ds[i][0] for i in range(N_EVAL)]).to(device)
for i in range(0, N_EVAL, 250):  # em lotes, para caber na memória
    fid.update(to_fid(real[i:i+250]), real=True)

@torch.no_grad()
def evaluate(sampler, **kw):
    y = torch.arange(10, device=device).repeat(N_EVAL // 10)
    outs = []
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for i in range(0, len(y), 500):
        outs.append(sampler(ema, y[i:i+500], seed=i, **kw)[0])
    if device == "cuda": torch.cuda.synchronize()
    dt = time.time() - t0
    x = torch.cat(outs)
    prob = clf(x).softmax(1)
    acc = (prob.argmax(1) == y).float().mean().item()
    conf = prob.max(1).values.mean().item()
    fid.reset()
    for i in range(0, len(x), 250):
        fid.update(to_fid(x[i:i+250]), real=False)
    return dict(acuracia=acc, confianca=conf, FID=fid.compute().item(), tempo_s=dt, imgs=x, y=y)

import pandas as pd
configs = [
    ("DDIM",  100, 3.0, lambda **k: sample_ddim(steps=100, **k)),
    ("DDIM",   50, 3.0, lambda **k: sample_ddim(steps=50, **k)),
    ("DDIM",   20, 3.0, lambda **k: sample_ddim(steps=20, **k)),
    ("DDIM",   10, 3.0, lambda **k: sample_ddim(steps=10, **k)),
    ("DDIM",    5, 3.0, lambda **k: sample_ddim(steps=5, **k)),
    ("DDIM",   50, 0.0, lambda **k: sample_ddim(steps=50, **k)),   # incondicional
    ("DDIM",   50, 1.0, lambda **k: sample_ddim(steps=50, **k)),   # só condicional
    ("DDIM",   50, 7.0, lambda **k: sample_ddim(steps=50, **k)),   # guidance forte
]
if AVALIAR_DDPM_1000:
    configs.insert(0, ("DDPM", 1000, 3.0, lambda **k: sample_ddpm(**k)))

rows, results = [], {}
for name, steps, w, fn in configs:
    r = evaluate(lambda m, y, seed: fn(model=m, y=y, guidance=w, seed=seed))
    results[(name, steps, w)] = r
    rows.append(dict(amostrador=name, passos=steps, guidance=w,
                     **{k: round(v, 4) for k, v in r.items() if k not in ("imgs", "y")}))
    print(rows[-1])
df = pd.DataFrame(rows); df.to_csv(f"{FIG_DIR}/metricas.csv", index=False); df

In [ ]:
#@title 8c. Gráficos da avaliação
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
d = df[(df.amostrador == "DDIM") & (df.guidance == 3.0)].sort_values("passos")
ax[0].plot(d.passos, d.FID, "o-"); ax[0].set_xscale("log"); ax[0].set_xlabel("passos (DDIM)"); ax[0].set_ylabel("FID ↓"); ax[0].set_title("Qualidade x nº de passos")
ax[1].plot(d.passos, d.tempo_s, "o-", color="tab:orange"); ax[1].set_xscale("log"); ax[1].set_xlabel("passos (DDIM)"); ax[1].set_ylabel("tempo (s)"); ax[1].set_title(f"Custo para {N_EVAL} imagens")
g = df[(df.amostrador == "DDIM") & (df.passos == 50)].sort_values("guidance")
ax[2].plot(g.guidance, g.acuracia, "o-", label="acurácia condicional"); ax2 = ax[2].twinx(); ax2.plot(g.guidance, g.FID, "s--", color="tab:red", label="FID")
ax[2].set_xlabel("guidance w"); ax[2].set_ylabel("acurácia"); ax2.set_ylabel("FID"); ax[2].set_title("Efeito do guidance (DDIM-50)")
plt.tight_layout(); salvar("10_graficos_avaliacao"); plt.show()

# Visual: mesma semente com diferentes nº de passos
row = []
for s in [5, 10, 20, 50, 100]:
    row.append(sample_ddim(ema, torch.arange(10, device=device), steps=s, guidance=3.0, seed=7)[0])
show(torch.cat(row), nrow=10, title="DDIM com 5, 10, 20, 50, 100 passos (linhas)", nome="11_passos_comparacao")

## 9. Erros observados
Mostramos as amostras em que o classificador **discorda** da classe pedida ou tem **baixa confiança** — são os casos de falha do gerador.

In [ ]:
#@title 9. Casos de falha
for key in [("DDIM", 50, 3.0), ("DDIM", 5, 3.0), ("DDIM", 50, 0.0)]:
    r = results[key]
    prob = clf(r["imgs"]).softmax(1)
    pred = prob.argmax(1)
    wrong = (pred != r["y"]).nonzero().flatten()
    print(f"{key}: {len(wrong)} de {len(r['y'])} amostras classificadas como outro dígito")
    if len(wrong):
        idx = wrong[:20]
        show(r["imgs"][idx], nrow=10, size=8, nome=f"12_falhas_{key[0]}{key[1]}_w{int(key[2])}",
             title=f"{key[0]}-{key[1]}, w={key[2]} | pedido→previsto: " +
                   " ".join(f"{a}→{b}" for a, b in zip(r['y'][idx].tolist(), pred[idx].tolist())))

# Matriz de confusão (DDIM-50, w=3)
from sklearn.metrics import ConfusionMatrixDisplay
r = results[("DDIM", 50, 3.0)]
ConfusionMatrixDisplay.from_predictions(r["y"].cpu(), clf(r["imgs"]).argmax(1).cpu(), cmap="Oranges")
plt.title("Classe pedida x classe reconhecida"); salvar("13_matriz_confusao"); plt.show()

## 10. Diversidade
O mesmo dígito com sementes diferentes deve gerar **estilos diferentes** (espessura, inclinação). Se todas as imagens forem iguais, houve *mode collapse* — um problema comum em GANs e raro em modelos de difusão.

In [ ]:
#@title 10. Diversidade: 16 versões do dígito escolhido
DIGITO = 4  #@param {type:"integer"}
imgs, _ = sample_ddim(ema, torch.full((16,), DIGITO, device=device), steps=50, guidance=3.0, seed=123)
show(imgs, nrow=8, title=f"16 amostras do dígito {DIGITO}", size=8, nome="14_diversidade")

## 11. Demonstração ao vivo — nosso modelo (Gradio)
Rode a célula **no Colab** e use o link público (`gradio.live`) que aparece. Dá para escolher o dígito, o amostrador, o nº de passos, o guidance e a semente.

In [ ]:
#@title 11. App interativo do DDPM treinado do zero
import gradio as gr

def gerar(digito, amostrador, passos, guidance, semente, quantidade):
    y = torch.full((int(quantidade),), int(digito), device=device)
    t0 = time.time()
    if amostrador == "DDPM (1000 passos)":
        x, frames = sample_ddpm(ema, y, guidance=guidance, save_every=100, seed=int(semente))
    else:
        x, frames = sample_ddim(ema, y, steps=int(passos), guidance=guidance, save_every=max(1, int(passos) // 10), seed=int(semente))
    dt = time.time() - t0
    up = lambda t: (torchvision.utils.make_grid((t + 1) / 2, nrow=8, padding=2, pad_value=1)
                    .permute(1, 2, 0).clamp(0, 1).numpy())
    final = np.kron(up(x.cpu()), np.ones((6, 6, 1)))
    cadeia = torch.cat([f[1][:1] for f in frames])
    cadeia = np.kron(up(cadeia), np.ones((4, 4, 1)))
    prob = clf(x).softmax(1)
    acc = (prob.argmax(1) == y).float().mean().item()
    return final, cadeia, f"Tempo: {dt:.2f}s | reconhecidas como {int(digito)} pelo classificador: {acc:.0%}"

demo = gr.Interface(
    fn=gerar,
    inputs=[gr.Slider(0, 9, 7, step=1, label="Dígito"),
            gr.Radio(["DDIM", "DDPM (1000 passos)"], value="DDIM", label="Amostrador"),
            gr.Slider(2, 200, 50, step=1, label="Passos (DDIM)"),
            gr.Slider(0, 10, 3, step=0.5, label="Guidance (w)"),
            gr.Number(42, label="Semente"),
            gr.Slider(1, 16, 8, step=1, label="Quantidade")],
    outputs=[gr.Image(label="Resultado"), gr.Image(label="Processo reverso (ruído → imagem)"), gr.Textbox(label="Métricas")],
    title="Diffusion Model treinado do zero — MNIST",
    description="DDPM condicional com Classifier-Free Guidance · SENAI CIMATEC")
demo.launch(share=True, debug=False)

## 12. Demonstração — Stable Diffusion (texto → imagem)
Para mostrar a técnica em escala real usamos o **SD-Turbo** (Stability AI), um *Latent Diffusion Model* destilado que gera 512×512 em **1–4 passos**.
Diferenças em relação ao nosso modelo: a difusão acontece no **espaço latente** de um VAE (64×64×4 em vez de pixels) e a condição é um **texto** codificado pelo CLIP, injetado na U-Net via *cross-attention*.

> Requer GPU. O download do modelo (~2,5 GB) leva 1–2 min. Se a T4 ficar sem memória, reinicie o ambiente e rode só esta seção.

In [ ]:
#@title 12a. Carregar o SD-Turbo
pipe = None
if device != "cuda":
    print("O SD-Turbo precisa de GPU: ative a GPU no Colab para rodar a Seção 12.")
else:
    from diffusers import AutoPipelineForText2Image
    pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.set_progress_bar_config(disable=True)

In [ ]:
#@title 12b. Gerar a partir de um prompt
prompt = "a robot reading a book in a library in Salvador, Bahia, watercolor"  #@param {type:"string"}
passos = 1  #@param {type:"slider", min:1, max:4, step:1}
if pipe is None:
    print("Seção 12 pulada: sem GPU.")
else:
    g = torch.Generator("cuda").manual_seed(0)
    t0 = time.time()
    imgs = pipe(prompt=prompt, num_inference_steps=passos, guidance_scale=0.0, num_images_per_prompt=4, generator=g).images
    print(f"{time.time()-t0:.2f}s para 4 imagens 512x512")
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    for a, im in zip(ax, imgs): a.imshow(im); a.axis("off")
    plt.suptitle(prompt); plt.show()

In [ ]:
#@title 12c. App Gradio — texto → imagem
def sd(prompt, passos, semente):
    g = torch.Generator("cuda").manual_seed(int(semente))
    t0 = time.time()
    im = pipe(prompt=prompt, num_inference_steps=int(passos), guidance_scale=0.0, generator=g).images[0]
    return im, f"{time.time()-t0:.2f}s"
if pipe is None:
    print("Seção 12 pulada: sem GPU.")
else:
    gr.Interface(sd,
                 [gr.Textbox("an astronaut playing berimbau on the beach, oil painting", label="Prompt (em inglês funciona melhor)"),
                  gr.Slider(1, 4, 1, step=1, label="Passos"), gr.Number(0, label="Semente")],
                 [gr.Image(label="Imagem"), gr.Textbox(label="Tempo")],
                 title="Stable Diffusion (SD-Turbo) — demonstração").launch(share=True)